# CNN Baseline Training

This notebook is set up to run end-to-end on a Google Colab runtime.

When it detects Colab, it will:
- clone the GitHub repo into `/content/speech-emotion-directions`
- install any missing training dependencies
- download the official RAVDESS speech archive if needed
- rebuild the metadata CSV with Colab file paths
- launch the speaker-independent CNN baseline training run

When it is run locally, it uses the current project folder instead.

<!-- narrative:05-role -->
## Notebook role in the project

This notebook trains the simpler CNN baseline on log-mel spectrograms. The baseline is important because it tells us whether wav2vec2 is actually adding value beyond a conventional acoustic model.

**Inputs:** the same RAVDESS metadata/audio as the main model.

**Outputs:** CNN checkpoint, validation/test metrics, predictions, and Drive backup.

**How to read it:** the CNN is not expected to win. It is a sanity baseline that helps justify focusing interpretability on the stronger wav2vec2 model.


In [1]:
from __future__ import annotations

import importlib.util
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"
RAVDESS_SOURCES = [
    {
        "name": "RAVDESS Speech 16K",
        "url": "https://zenodo.org/api/records/11063852/files/Audio_Speech_Actors_01-24_16k.zip/content",
        "archive_name": "Audio_Speech_Actors_01-24_16k.zip",
    },
    {
        "name": "RAVDESS Speech Original",
        "url": "https://zenodo.org/api/records/1188976/files/Audio_Speech_Actors_01-24.zip/content",
        "archive_name": "Audio_Speech_Actors_01-24.zip",
    },
]

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)


def run_command(cmd: list[str], cwd: Path | None = None) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)



def ensure_colab_packages() -> None:
    if not IS_COLAB:
        return

    package_map = {
        "yaml": "pyyaml",
        "pandas": "pandas",
        "numpy": "numpy",
        "soundfile": "soundfile",
        "librosa": "librosa",
        "torch": "torch",
        "torchaudio": "torchaudio",
        "transformers": "transformers",
        "datasets": "datasets",
        "huggingface_hub": "huggingface_hub",
        "accelerate": "accelerate",
        "sklearn": "scikit-learn",
        "tqdm": "tqdm",
    }
    missing_packages = sorted({pkg for module, pkg in package_map.items() if importlib.util.find_spec(module) is None})
    if missing_packages:
        run_command([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
    else:
        print("Required training packages are already available.")



def find_local_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / "FinalProject"]

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "configs" / "wav2vec.yaml").exists() and (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root locally. Expected a folder containing configs/wav2vec.yaml and src/."
    )



def repo_status_lines(repo_root: Path) -> list[str]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "status", "--short"],
        text=True,
        capture_output=True,
        check=True,
    )
    return [line for line in result.stdout.splitlines() if line.strip()]



def repo_ahead_behind(repo_root: Path) -> tuple[int, int]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "rev-list", "--left-right", "--count", "HEAD...origin/main"],
        text=True,
        capture_output=True,
        check=False,
    )
    if result.returncode != 0 or not result.stdout.strip():
        return 0, 0
    ahead_str, behind_str = result.stdout.strip().split()
    return int(ahead_str), int(behind_str)



def clone_clean_code_checkout(clean_root: Path) -> Path:
    if clean_root.exists():
        shutil.rmtree(clean_root)
    run_command(["git", "clone", "--depth", "1", REPO_URL, str(clean_root)])
    return clean_root



def prepare_project_and_code_roots() -> tuple[Path, Path]:
    runtime_root = Path("/content") / REPO_NAME
    clean_root = Path("/content") / f"{REPO_NAME}-clean"

    if runtime_root.exists() and (runtime_root / ".git").exists():
        try:
            run_command(["git", "-C", str(runtime_root), "fetch", "origin"])
        except subprocess.CalledProcessError as exc:
            print(f"Fetch failed for existing Colab repo; continuing with local state. Details: {exc}")

        status_lines = repo_status_lines(runtime_root)
        ahead, behind = repo_ahead_behind(runtime_root)

        if not status_lines and ahead == 0:
            if behind > 0:
                try:
                    run_command(["git", "-C", str(runtime_root), "pull", "--ff-only", "origin", "main"])
                    code_root = runtime_root
                except subprocess.CalledProcessError as exc:
                    print(f"Fast-forward pull failed; using a clean code checkout instead. Details: {exc}")
                    code_root = clone_clean_code_checkout(clean_root)
            else:
                code_root = runtime_root
        else:
            print(f"Using a clean code checkout because {runtime_root} has local changes or local commits.")
            for line in status_lines[:10]:
                print(" -", line)
            if ahead or behind:
                print(f"Repo divergence relative to origin/main: ahead={ahead}, behind={behind}")
            code_root = clone_clean_code_checkout(clean_root)

        project_root = runtime_root
    elif runtime_root.exists():
        print(f"Using existing non-git project directory for data/artifacts: {runtime_root}")
        project_root = runtime_root
        code_root = clone_clean_code_checkout(clean_root)
    else:
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(runtime_root)])
        project_root = runtime_root
        code_root = runtime_root

    return project_root, code_root


ensure_colab_packages()

if IS_COLAB:
    PROJECT_ROOT, CODE_ROOT = prepare_project_and_code_roots()
else:
    PROJECT_ROOT = find_local_project_root()
    CODE_ROOT = PROJECT_ROOT

def module_uses_code_root(module_file: str | None, code_root: Path) -> bool:
    if module_file is None:
        return False
    try:
        Path(module_file).resolve().relative_to(code_root.resolve())
        return True
    except ValueError:
        return False


os.chdir(PROJECT_ROOT)
for root in [str(CODE_ROOT), str(PROJECT_ROOT)]:
    while root in sys.path:
        sys.path.remove(root)

sys.path.insert(0, str(CODE_ROOT))
if str(PROJECT_ROOT) != str(CODE_ROOT):
    sys.path.insert(1, str(PROJECT_ROOT))

src_module = sys.modules.get("src")
src_file = getattr(src_module, "__file__", None)
if src_module is not None and not module_uses_code_root(src_file, CODE_ROOT):
    stale_modules = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
    for name in stale_modules:
        del sys.modules[name]

print("Project root:", PROJECT_ROOT)
print("Code root:", CODE_ROOT)
print("Working directory:", Path.cwd().resolve())

Running in Colab: True
Required training packages are already available.
+ git -C /content/speech-emotion-directions fetch origin
Using a clean code checkout because /content/speech-emotion-directions has local changes or local commits.
 -  M data/metadata/ravdess_metadata.csv
+ git clone --depth 1 https://github.com/pavannn16/speech-emotion-directions.git /content/speech-emotion-directions-clean
Project root: /content/speech-emotion-directions
Code root: /content/speech-emotion-directions-clean
Working directory: /content/speech-emotion-directions


<!-- narrative:05-data-cache -->
## Shared dataset setup

The baseline uses the same metadata and speaker-independent split as wav2vec2. Keeping the data setup identical is what makes the later model comparison fair.


In [2]:
import pandas as pd
from IPython.display import display

from huggingface_hub import snapshot_download

from src.data.ravdess_metadata import build_ravdess_metadata, save_metadata

raw_root = PROJECT_ROOT / "data" / "raw"
extract_dir = raw_root / "ravdess_audio_speech"
metadata_path = PROJECT_ROOT / "data" / "metadata" / "ravdess_metadata.csv"

drive_dataset_dir = (
    Path("/content/drive/MyDrive") / "speech-emotion-directions" / "datasets" / "ravdess_audio_speech"
    if IS_COLAB
    else None
)

raw_root.mkdir(parents=True, exist_ok=True)
metadata_path.parent.mkdir(parents=True, exist_ok=True)


def count_wavs(root: Path | None) -> int:
    if root is None or not root.exists():
        return 0
    return sum(1 for _ in root.rglob("*.wav"))


def restore_ravdess_from_drive_cache() -> bool:
    """Use the Google Drive dataset cache as the first persistent source in Colab."""
    if drive_dataset_dir is None:
        return False

    drive_count = count_wavs(drive_dataset_dir)
    print("Drive RAVDESS wav cache:", drive_dataset_dir, f"({drive_count} wavs)")
    if drive_count < 1440:
        return False

    local_count = count_wavs(extract_dir)
    if local_count >= 1440:
        print("Local runtime already has complete RAVDESS audio; Drive cache is available as the persistent source.")
        return True

    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(drive_dataset_dir, extract_dir)
    print("Restored RAVDESS audio from Google Drive cache to:", extract_dir)
    return True


def cache_ravdess_to_drive() -> None:
    """Persist the local RAVDESS extract to Drive for later notebooks/runtimes."""
    if drive_dataset_dir is None:
        return

    local_count = count_wavs(extract_dir)
    if local_count < 1440:
        print(f"Skipping Drive dataset cache because local audio is incomplete ({local_count} wavs).")
        return

    drive_count = count_wavs(drive_dataset_dir)
    if drive_count >= 1440:
        print("Google Drive RAVDESS cache already complete:", drive_dataset_dir)
        return

    if drive_dataset_dir.exists():
        shutil.rmtree(drive_dataset_dir)
    drive_dataset_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(extract_dir, drive_dataset_dir)
    print("Cached RAVDESS audio in Google Drive:", drive_dataset_dir)


def download_file(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        destination.unlink()

    commands = [
        [
            "curl", "-L", "--fail", "--retry", "3", "--retry-delay", "5",
            "-A", "Mozilla/5.0", "-o", str(destination), url,
        ],
        [
            "wget", "--tries=3", "--waitretry=5", "--user-agent=Mozilla/5.0",
            "-O", str(destination), url,
        ],
    ]

    last_error = None
    for command in commands:
        if shutil.which(command[0]) is None:
            continue
        try:
            print("+", " ".join(command))
            subprocess.check_call(command)
            if destination.exists() and destination.stat().st_size > 0:
                return
        except subprocess.CalledProcessError as exc:
            last_error = exc
            if destination.exists():
                destination.unlink()

    raise RuntimeError(f"Failed to download dataset from {url}") from last_error


def ensure_ravdess_archive(raw_root: Path) -> Path:
    last_error = None
    for source in RAVDESS_SOURCES:
        archive_path = raw_root / source["archive_name"]
        if archive_path.exists() and archive_path.stat().st_size > 0:
            print(f"Using existing archive: {archive_path}")
            return archive_path

        try:
            print(f"Downloading {source['name']}...")
            download_file(source["url"], archive_path)
            print(f"Downloaded archive to: {archive_path}")
            return archive_path
        except RuntimeError as exc:
            print(f"Download failed for {source['name']}: {exc}")
            last_error = exc

    raise RuntimeError("Unable to download any RAVDESS speech archive in this runtime.") from last_error


def download_from_hf_mirror(destination_root: Path) -> None:
    print("Falling back to the Hugging Face RAVDESS mirror...")
    snapshot_download(
        repo_id="birgermoell/ravdess",
        repo_type="dataset",
        local_dir=destination_root,
        allow_patterns=["Actor_*/*.wav"],
    )


def ensure_ravdess_audio() -> None:
    print("Existing local extracted wav files:", count_wavs(extract_dir))

    if restore_ravdess_from_drive_cache():
        return

    if count_wavs(extract_dir) >= 1440:
        cache_ravdess_to_drive()
        return

    archive_downloaded = False
    try:
        archive_path = ensure_ravdess_archive(raw_root)
        archive_downloaded = True
    except RuntimeError as exc:
        print(exc)

    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    if archive_downloaded:
        print("Extracting archive...")
        with zipfile.ZipFile(archive_path, "r") as archive:
            archive.extractall(extract_dir)
    else:
        download_from_hf_mirror(extract_dir)

    cache_ravdess_to_drive()


ensure_ravdess_audio()
wav_count = count_wavs(extract_dir)
print("Extracted wav files:", wav_count)
assert wav_count >= 1440, f"Expected at least 1440 speech wav files, found {wav_count}."

df = build_ravdess_metadata(extract_dir)
save_metadata(df, metadata_path)

project_df = df[df["keep_for_project"]].copy()
print("Metadata saved to:", metadata_path)
print("Total speech clips:", len(df))
print("Project clips:", len(project_df))
display(project_df["final_label"].value_counts().sort_index().rename("count").to_frame())
display(project_df.groupby(["split", "final_label"]).size().unstack(fill_value=0))


Existing local extracted wav files: 1440
Drive RAVDESS wav cache: /content/drive/MyDrive/speech-emotion-directions/datasets/ravdess_audio_speech (1440 wavs)
Local runtime already has complete RAVDESS audio; Drive cache is available as the persistent source.
Extracted wav files: 1440
Metadata saved to: /content/speech-emotion-directions/data/metadata/ravdess_metadata.csv
Total speech clips: 1440
Project clips: 1248


,count
final_label,
angry,192
disgust,192
fearful,192
happy,192
neutral,288
sad,192


final_label,angry,disgust,fearful,happy,neutral,sad
split,,,,,,
test,32,32,32,32,48,32
train,128,128,128,128,192,128
val,32,32,32,32,48,32


<!-- narrative:05-config -->
## CNN baseline configuration

This cell loads the CNN training settings. Unlike wav2vec2, the CNN receives log-mel spectrogram features, so it tests a more traditional speech-emotion pipeline.


In [3]:
import json

import torch

from src.utils.config import load_yaml_config

config_path = CODE_ROOT / 'configs' / 'cnn_baseline.yaml'
config = load_yaml_config(config_path)

config['metadata_path'] = str(metadata_path.resolve())
config['output_dir'] = str((PROJECT_ROOT / config['output_dir']).resolve())

if IS_COLAB and torch.cuda.is_available():
    config['batch_size'] = 32
    config['eval_batch_size'] = 64
    config['num_workers'] = 2

output_dir = Path(config['output_dir'])
output_dir.parent.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device_summary = torch.cuda.get_device_name(0)
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device_summary = 'Apple Metal (MPS)'
else:
    device_summary = 'CPU'

assert config_path.exists(), f'Config file not found: {config_path}'
assert metadata_path.exists(), f'Metadata file not found: {metadata_path}'

print('Config file:', config_path)
print('Metadata file:', metadata_path)
print('Output directory:', output_dir)
print('Detected device:', device_summary)
print(json.dumps(config, indent=2))

Config file: /content/speech-emotion-directions-clean/configs/cnn_baseline.yaml
Metadata file: /content/speech-emotion-directions/data/metadata/ravdess_metadata.csv
Output directory: /content/speech-emotion-directions/artifacts/checkpoints/cnn_baseline_ravdess_speaker_independent_v1
Detected device: NVIDIA A100-SXM4-40GB
{
  "experiment_name": "cnn_baseline_ravdess_speaker_independent_v1",
  "metadata_path": "/content/speech-emotion-directions/data/metadata/ravdess_metadata.csv",
  "output_dir": "/content/speech-emotion-directions/artifacts/checkpoints/cnn_baseline_ravdess_speaker_independent_v1",
  "sample_rate": 16000,
  "n_mels": 64,
  "n_fft": 400,
  "hop_length": 160,
  "win_length": 400,
  "fmin": 0.0,
  "fmax": 8000.0,
  "target_frames": 600,
  "batch_size": 32,
  "eval_batch_size": 64,
  "num_workers": 2,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "num_epochs": 30,
  "early_stopping_patience": 6,
  "dropout": 0.3,
  "hidden_dim": 128,
  "seed": 42,
  "use_class_weigh

<!-- narrative:05-training-call -->
## Training the baseline

This is the CNN training call. The output metrics will later be compared against wav2vec2 in notebook 06. The baseline gives us a practical reference point: if the main model only barely beat this model, the interpretability story would be less convincing.


In [4]:
from src.training.train_cnn_baseline import run_training

run_training(config, dry_run=False)

epoch=1 train_loss=1.7617 val_accuracy=0.2308 val_macro_f1=0.0637


epoch=2 train_loss=1.6676 val_accuracy=0.3317 val_macro_f1=0.1611


epoch=3 train_loss=1.6001 val_accuracy=0.3606 val_macro_f1=0.2803


epoch=4 train_loss=1.5159 val_accuracy=0.3558 val_macro_f1=0.2036


epoch=5 train_loss=1.4744 val_accuracy=0.1683 val_macro_f1=0.0702


...

epoch=15 train_loss=1.2960 val_accuracy=0.2356 val_macro_f1=0.0730


epoch=16 train_loss=1.2690 val_accuracy=0.3173 val_macro_f1=0.1993


epoch=17 train_loss=1.2899 val_accuracy=0.2308 val_macro_f1=0.1664


epoch=18 train_loss=1.2610 val_accuracy=0.3221 val_macro_f1=0.2410
Early stopping triggered.


Saved model and evaluation artifacts to /content/speech-emotion-directions/artifacts/checkpoints/cnn_baseline_ravdess_speaker_independent_v1
Final val macro F1: 0.3722
Final test macro F1: 0.3241


<!-- narrative:05-read-metrics -->
## Reading baseline outputs

These printed metrics are kept in the notebook so GitHub preserves the baseline evidence. The important comparison is the test macro F1 gap between wav2vec2 and the CNN.


In [5]:
import json
import pandas as pd
from IPython.display import display

print('Output directory:', output_dir)
print('Exists:', output_dir.exists())

if output_dir.exists():
    for path in sorted(output_dir.iterdir()):
        print(path.name)

for metrics_name in ['val_metrics.json', 'test_metrics.json']:
    metrics_path = output_dir / metrics_name
    if metrics_path.exists():
        with metrics_path.open('r', encoding='utf-8') as handle:
            metrics = json.load(handle)
        summary = {
            'accuracy': metrics.get('accuracy'),
            'macro_f1': metrics.get('macro_f1'),
            'weighted_f1': metrics.get('weighted_f1'),
        }
        print(f"\n{metrics_name}")
        print(json.dumps(summary, indent=2))

report_path = output_dir / 'test_classification_report.csv'
predictions_path = output_dir / 'test_predictions.csv'
if report_path.exists():
    display(pd.read_csv(report_path).head(10))
if predictions_path.exists():
    display(pd.read_csv(predictions_path).head(5))

Output directory: /content/speech-emotion-directions/artifacts/checkpoints/cnn_baseline_ravdess_speaker_independent_v1
Exists: True
config.json
label_mapping.json
model_state.pt
test_classification_report.csv
test_metrics.json
test_predictions.csv
val_metrics.json

val_metrics.json
{
  "accuracy": 0.4375,
  "macro_f1": 0.37218226812361666,
  "weighted_f1": 0.39122350210327345
}

test_metrics.json
{
  "accuracy": 0.4230769230769231,
  "macro_f1": 0.32411116130628326,
  "weighted_f1": 0.3475926588496945
}


,Unnamed: 0,precision,recall,f1-score,support
0,neutral,0.473684,0.937500,0.629371,48.000000
1,happy,0.111111,0.031250,0.048780,32.000000
2,sad,0.125000,0.031250,0.050000,32.000000
3,angry,0.666667,0.187500,0.292683,32.000000
4,fearful,0.333333,0.437500,0.378378,32.000000
5,disgust,0.466667,0.656250,0.545455,32.000000
6,accuracy,0.423077,0.423077,0.423077,0.423077
7,macro avg,0.362744,0.380208,0.324111,208.000000
8,weighted avg,0.371278,0.423077,0.347593,208.000000


,file_name,file_path,actor_id,statement_code,statement,emotion,final_label,intensity,split,duration_seconds,true_label_id,pred_label_id,true_label,pred_label,prob_neutral,prob_happy,prob_sad,prob_angry,prob_fearful,prob_disgust
0,03-01-01-01-01-01-21.wav,/content/speech-emotion-directions/data/raw/ra...,21,1,Kids are talking by the door,neutral,neutral,normal,test,3.837167,0,0,neutral,neutral,0.632081,0.023207,0.117804,0.014704,0.004561,0.207644
1,03-01-01-01-01-02-21.wav,/content/speech-emotion-directions/data/raw/ra...,21,1,Kids are talking by the door,neutral,neutral,normal,test,3.970625,0,0,neutral,neutral,0.661560,0.014679,0.092364,0.010828,0.002050,0.218520
2,03-01-01-01-02-01-21.wav,/content/speech-emotion-directions/data/raw/ra...,21,2,Dogs are sitting by the door,neutral,neutral,normal,test,3.603604,0,0,neutral,neutral,0.493256,0.076208,0.189399,0.030964,0.030497,0.179676
3,03-01-01-01-02-02-21.wav,/content/speech-emotion-directions/data/raw/ra...,21,2,Dogs are sitting by the door,neutral,neutral,normal,test,3.670333,0,0,neutral,neutral,0.643894,0.031494,0.146603,0.011990,0.010666,0.155353
4,03-01-02-01-01-01-21.wav,/content/speech-emotion-directions/data/raw/ra...,21,1,Kids are talking by the door,calm,neutral,normal,test,4.037375,0,0,neutral,neutral,0.806671,0.006808,0.095711,0.001894,0.001968,0.086948


## Optional Google Drive Sync

This sync step is the recommended way to preserve the full training outputs from Colab.

It copies the complete fine-tuned checkpoint directory, the metadata CSV, and a short run summary into your Google Drive under:

`MyDrive/speech-emotion-directions/runs/<experiment_name>/`

That makes it easy to keep the large model artifacts without trying to force them into GitHub.

In [6]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

if not IS_COLAB:
    print("Google Drive sync is intended for Colab runtimes. Skipping on local execution.")
else:
    from google.colab import drive  # type: ignore

    drive.mount('/content/drive', force_remount=False)

    drive_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' / config['experiment_name']
    checkpoint_backup_dir = drive_root / 'checkpoint'
    metadata_backup_dir = drive_root / 'metadata'

    assert output_dir.exists(), f"Training output directory does not exist: {output_dir}"
    assert metadata_path.exists(), f"Metadata CSV does not exist: {metadata_path}"

    if checkpoint_backup_dir.exists():
        shutil.rmtree(checkpoint_backup_dir)
    checkpoint_backup_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(output_dir, checkpoint_backup_dir)

    metadata_backup_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(metadata_path, metadata_backup_dir / metadata_path.name)

    val_metrics = {}
    test_metrics = {}
    if (output_dir / 'val_metrics.json').exists():
        val_metrics = json.loads((output_dir / 'val_metrics.json').read_text(encoding='utf-8'))
    if (output_dir / 'test_metrics.json').exists():
        test_metrics = json.loads((output_dir / 'test_metrics.json').read_text(encoding='utf-8'))

    summary_lines = [
        f"# {config['experiment_name']}",
        "",
        f"Generated at: {datetime.now(timezone.utc).isoformat()}",
        f"Colab project root: {PROJECT_ROOT}",
        f"Checkpoint source: {output_dir}",
        f"Drive backup root: {drive_root}",
        "",
        "## Validation Metrics",
        f"- Accuracy: {val_metrics.get('accuracy')}",
        f"- Macro F1: {val_metrics.get('macro_f1')}",
        f"- Weighted F1: {val_metrics.get('weighted_f1')}",
        "",
        "## Test Metrics",
        f"- Accuracy: {test_metrics.get('accuracy')}",
        f"- Macro F1: {test_metrics.get('macro_f1')}",
        f"- Weighted F1: {test_metrics.get('weighted_f1')}",
        "",
        "This Drive backup contains the full fine-tuned checkpoint and metadata used for the run.",
    ]
    summary_path = drive_root / 'run_summary.md'
    summary_path.write_text("\n".join(summary_lines) + "\n", encoding='utf-8')

    print('Backed up training artifacts to Google Drive:')
    print(' -', checkpoint_backup_dir)
    print(' -', metadata_backup_dir / metadata_path.name)
    print(' -', summary_path)

    print('\nDrive backup contents:')
    for path in sorted(drive_root.rglob('*')):
        relative = path.relative_to(drive_root)
        if path.is_dir():
            print(f' [dir]  {relative}')
        else:
            print(f' [file] {relative}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Backed up training artifacts to Google Drive:
 - /content/drive/MyDrive/speech-emotion-directions/runs/cnn_baseline_ravdess_speaker_independent_v1/checkpoint
 - /content/drive/MyDrive/speech-emotion-directions/runs/cnn_baseline_ravdess_speaker_independent_v1/metadata/ravdess_metadata.csv
 - /content/drive/MyDrive/speech-emotion-directions/runs/cnn_baseline_ravdess_speaker_independent_v1/run_summary.md

Drive backup contents:
 [dir]  checkpoint
 [file] checkpoint/config.json
 [file] checkpoint/label_mapping.json
 [file] checkpoint/model_state.pt
 [file] checkpoint/test_classification_report.csv
 [file] checkpoint/test_metrics.json
 [file] checkpoint/test_predictions.csv
 [file] checkpoint/val_metrics.json
 [dir]  metadata
 [file] metadata/ravdess_metadata.csv
 [file] run_summary.md
